# Demo 2 — One question, three ways

> *How many open issues does this course's repository have, and when was it last
> pushed to?*

Three ways to answer it, and the differences are the whole point.

| | |
|---|---|
| **a prompt** | ask a model |
| **an API** | call the service yourself |
| **an MCP tool** | let the agent find the call |

Runs offline.

In [3]:
# Setup. Works from anywhere inside the course checkout.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise SystemExit(f"No course found above {Path.cwd()}. Open this inside your checkout.")
sys.path.insert(0, str(REPO_ROOT / "src"))
print("ready")

ready


## 1. Ask a model

A model is trained once and then frozen. This question is about the world *now*.

In [1]:
from bootcamp_agent.llm import FakeLLM

QUESTION = (
    "How many open issues does Gecko-Academy/dev3pack-cohort-2026-09 have "
    "right now, and when was it last pushed to?"
)

model = FakeLLM(default=(
    "I don't have access to live data, so I can't tell you the current issue "
    "count or the last push time for that repository."
))
print(model.complete(system="Be concise.", user=QUESTION))

I don't have access to live data, so I can't tell you the current issue count or the last push time for that repository.


**That is the correct answer**, and it is worth dwelling on. The model cannot
know: the number changes by the hour and the weights are months old.

A worse model invents a number instead. Same question, confident reply, and
nothing on the outside tells the two apart. That is the failure mode — not
ignorance, but *fluent* ignorance.

## 2. Call the API

Now you get a real answer. Notice what it cost.

In [2]:
import json
import urllib.error
import urllib.request

RECORDED = {"open_issues_count": 0, "pushed_at": "2026-09-16T04:00:00Z"}


def repo_facts() -> tuple[dict, str]:
    request = urllib.request.Request(
        "https://api.github.com/repos/Gecko-Academy/dev3pack-cohort-2026-09",
        headers={"Accept": "application/vnd.github+json", "User-Agent": "dev3pack-demo"},
    )
    try:
        with urllib.request.urlopen(request, timeout=10) as response:
            return json.loads(response.read().decode("utf-8")), "live"
    except (urllib.error.URLError, TimeoutError, OSError):
        return RECORDED, "recorded"


data, source = repo_facts()
print(f"[{source}] open issues: {data['open_issues_count']}")
print(f"[{source}] last push:   {data['pushed_at']}")

[live] open issues: 0
[live] last push:   2026-09-18T05:13:07Z


To write those nine lines you had to know four things nobody told the model:

1. the host (`api.github.com`)
2. the path shape (`/repos/{owner}/{repo}`)
3. that GitHub refuses a request with no `User-Agent`
4. which two of the ~80 fields in the reply you wanted

**You read the documentation.** That is the cost, and it is paid per API, by a
human, every time.

## 3. The same call, described

An MCP tool is that same HTTP call, plus a description of what it is for and
what its arguments mean — in a form a model can *choose from* without reading
anything.

Below is a tool definition. Read it as the model would.

In [3]:
TOOL = {
    "name": "repo_activity",
    "description": (
        "How many issues are open on a GitHub repository right now, and when it "
        "was last pushed to. Use this for questions about a repository's CURRENT "
        "state, which a language model cannot know."
    ),
    "inputSchema": {
        "type": "object",
        "properties": {
            "owner": {"type": "string", "description": "The org or user, e.g. Gecko-Academy."},
            "repo": {"type": "string", "description": "The repository name."},
        },
        "required": ["owner", "repo"],
    },
}


def repo_activity(owner: str, repo: str) -> dict:
    """The same HTTP call as section 2, behind the contract above."""
    data, source = repo_facts()
    return {"open_issues": data["open_issues_count"], "last_push": data["pushed_at"],
            "source": source}


print(json.dumps(TOOL, indent=2))
print()
print("calling it:", repo_activity(owner="Gecko-Academy", repo="dev3pack-cohort-2026-09"))

{
  "name": "repo_activity",
  "description": "How many issues are open on a GitHub repository right now, and when it was last pushed to. Use this for questions about a repository's CURRENT state, which a language model cannot know.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "owner": {
        "type": "string",
        "description": "The org or user, e.g. Gecko-Academy."
      },
      "repo": {
        "type": "string",
        "description": "The repository name."
      }
    },
    "required": [
      "owner",
      "repo"
    ]
  }
}

calling it: {'open_issues': 0, 'last_push': '2026-09-18T05:13:07Z', 'source': 'live'}


## 4. So what is each one for

| | good at | never |
|---|---|---|
| **a prompt** | language, judgement, shape | a live fact |
| **an API** | the fact — for someone who read the docs | self-describing |
| **an MCP tool** | the same fact, described so a model can pick it | magic |

**MCP is not a faster API.** Underneath, section 3 makes the identical HTTP call
as section 2. The difference is the first three lines: the tool arrived carrying
its own description and the names of its arguments.

That is the whole idea, and sessions 9 to 13 build one.

In [4]:
print("Underneath, sections 2 and 3 made the same request. Compare:\n")
print("  section 2:  you knew the host, the path, the header, and the field")
print("  section 3:  the tool said what it was for, and the model picked it")
print()
print("Same network call. Different amount of human reading.")

Underneath, sections 2 and 3 made the same request. Compare:

  section 2:  you knew the host, the path, the header, and the field
  section 3:  the tool said what it was for, and the model picked it

Same network call. Different amount of human reading.
